In [ ]:
"""
ISYE 6420 Spring 2026 Final Project
Author: Foster Sicotte
 
Does Winning the Pistol Round Predict Winning the Map?
A Bayesian Logistic Regression Analysis of Professional Counter-Strike
 
This script:
1. Loads results.csv and economy.csv
2. Merges them and subsets to close-ranked matches (rank gap < 15) (we want teams that are closely ranked in skill gap) 
3. Fits three models:
   (a) Bayesian logistic regression with informative priors
   (b) Bayesian logistic regression with vague priors
   (c) Frequentist logistic regression (for comparison)
4. Generates posterior summaries, trace plots, and posterior predictive checks
5. Saves all outputs to the ./outputs/ directory
"""

'\nISyE 6420 Spring 2026 Final Project\nAuthor: Foster Sicotte\n\nDoes Winning the Pistol Round Predict Winning the Map?\nA Bayesian Logistic Regression Analysis of Professional Counter-Strike\n\nThis script:\n1. Loads results.csv and economy.csv\n2. Merges them and subsets to close-ranked matches (rank gap < 15)\n3. Fits three models:\n   (a) Bayesian logistic regression with informative priors\n   (b) Bayesian logistic regression with vague priors\n   (c) Frequentist logistic regression (for comparison)\n4. Generates posterior summaries, trace plots, and posterior predictive checks\n5. Saves all outputs to the ./outputs/ directory\n'

In [4]:
!pip install scikit-learn

   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 8.1/8.1 MB 71.7 MB/s  0:00:00

   ---------------------------------------- 0/2 [joblib]
   ---------------------------------------- 0/2 [joblib]
   ---------------------------------------- 0/2 [joblib]
   ---------------------------------------- 0/2 [joblib]
   ---------------------------------------- 0/2 [joblib]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -

In [6]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
 
import pymc as pm
import arviz as az
 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
 
warnings.filterwarnings("ignore")
RANDOM_SEED = 6420
rng = np.random.default_rng(RANDOM_SEED)
 
os.makedirs("outputs", exist_ok=True)

In [7]:
# 1. Load and prepare the data (we have two CSVs to blend) - ONLY USE results.csv and economy.csv 
print("Loading data...")
results = pd.read_csv("results.csv")
econ = pd.read_csv("economy.csv", low_memory=False)
 
# Join with match_id and map - this ensures we are using the same professional matches from each sheet 
df = econ[["match_id", "_map", "1_winner", "16_winner"]].merge(
    results[[
        "match_id", "_map", "team_1", "team_2", "map_winner",
        "rank_1", "rank_2", "starting_ct"
    ]],
    on=["match_id", "_map"],
    how="inner"
)
 
# Subset to closely ranked matches - We only want to use teams that are closely ranked to each other. This reduces the amount of rows/data needed, plus it also reduces any large skill gaps
df["rank_gap"] = (df["rank_1"] - df["rank_2"]).abs()
df["rank_diff"] = df["rank_2"] - df["rank_1"]  # positive => team 1 better ranked
 
close = df[df["rank_gap"] < 15].copy()
close = close.dropna(subset=["1_winner", "16_winner"])
 
# modeling variables
close["team1_won"] = (close["map_winner"] == 1).astype(int)
close["t1_won_pistol1"] = (close["1_winner"] == 1).astype(int)
close["t1_won_pistol2"] = (close["16_winner"] == 1).astype(int)
close["t1_started_ct"] = (close["starting_ct"] == 1).astype(int)
 
# Standardize rank_diff so prior scale is interpretable
close["rank_diff_std"] = (close["rank_diff"] - close["rank_diff"].mean()) / close["rank_diff"].std()
 
print(f"Final dataset: {len(close)} rows")
print(f"Team 1 win rate: {close['team1_won'].mean():.3f}")
 
#cleaned dataset output to save directory
close.to_csv("outputs/cleaned_data.csv", index=False)

Loading data...
Final dataset: 10581 rows
Team 1 win rate: 0.513


In [8]:
# 2.  statistics and summary data for the writeup portion 


print("\n=== Descriptive Rates ===")
rates = {
    "Won pistol 1 -> win map":
        close[close["t1_won_pistol1"] == 1]["team1_won"].mean(),
    "Lost pistol 1 -> win map":
        close[close["t1_won_pistol1"] == 0]["team1_won"].mean(),
    "Won pistol 2 -> win map":
        close[close["t1_won_pistol2"] == 1]["team1_won"].mean(),
    "Lost pistol 2 -> win map":
        close[close["t1_won_pistol2"] == 0]["team1_won"].mean(),
    "Won both pistols -> win map":
        close[(close["t1_won_pistol1"] == 1) & (close["t1_won_pistol2"] == 1)]["team1_won"].mean(),
    "Lost both pistols -> win map":
        close[(close["t1_won_pistol1"] == 0) & (close["t1_won_pistol2"] == 0)]["team1_won"].mean(),
}
for k, v in rates.items():
    print(f"  {k}: {v:.3f}")
 


=== Descriptive Rates ===
  Won pistol 1 -> win map: 0.613
  Lost pistol 1 -> win map: 0.411
  Won pistol 2 -> win map: 0.606
  Lost pistol 2 -> win map: 0.417
  Won both pistols -> win map: 0.707
  Lost both pistols -> win map: 0.314


In [10]:
# 3. Prepare design matrix

X = close[["t1_won_pistol1", "t1_won_pistol2", "rank_diff_std", "t1_started_ct"]].values
y = close["team1_won"].values
 
p1 = close["t1_won_pistol1"].values
p2 = close["t1_won_pistol2"].values
rd = close["rank_diff_std"].values
ct = close["t1_started_ct"].values

In [ ]:
# 4. Bayesian model with INFORMATIVE priors

# Priors reasoning:
#   beta0 (intercept): Normal(0, 1.5) centered at log odds 0 (50/50) since
#   outcome is balanced, moderate uncertainty.


#   beta_pistol1, beta_pistol2: Normal(1.0, 0.5) the community believes
#   pistol wins matter a lot. A coefficient of 1.0 on log-odds translates
#   to an odds ratio of e^1 ~= 2.7, meaning winning the pistol makes you roughly 2.7x more likely to win the map. Std 0.5 reflects moderate confidence in this belief.
#   beta_rankdiff: Normal(0.5, 0.3)  we know the higher-ranked team has an edge. rank_diff is standardized so the coefficient is per standard deviation. A 0.5 effect is a reasonable starting belief.
#   beta_ct: Normal(0, 0.3) the community also believes CT side is
#    favored, but our earlier descriptive look showed essentially no
#    effect at the match level. We set a skeptical prior centered at 0.
 
print("\n=== Fitting Bayesian model with informative priors ===")
with pm.Model() as model_informative:
    beta0 = pm.Normal("beta0", mu=0.0, sigma=1.5)
    beta_p1 = pm.Normal("beta_p1", mu=1.0, sigma=0.5)
    beta_p2 = pm.Normal("beta_p2", mu=1.0, sigma=0.5)
    beta_rd = pm.Normal("beta_rd", mu=0.5, sigma=0.3)
    beta_ct = pm.Normal("beta_ct", mu=0.0, sigma=0.3)
 
    logit_p = (beta0
               + beta_p1 * p1
               + beta_p2 * p2
               + beta_rd * rd
               + beta_ct * ct)
 
    y_obs = pm.Bernoulli("y_obs", logit_p=logit_p, observed=y)
 
    idata_inf = pm.sample(
        draws=2000,
        tune=1000,
        chains=4,
        random_seed=RANDOM_SEED,
        target_accept=0.9,
        return_inferencedata=True,
        progressbar=True,
    )
 
    # Posterior predictive
    pm.sample_posterior_predictive(idata_inf, extend_inferencedata=True,
                                   random_seed=RANDOM_SEED)
 
print(az.summary(idata_inf, var_names=["beta0", "beta_p1", "beta_p2",
                                       "beta_rd", "beta_ct"]))


=== Fitting Bayesian model with informative priors ===


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta0, beta_p1, beta_p2, beta_rd, beta_ct]


Output()

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 31 seconds.
Sampling: [y_obs]


Output()

          mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
beta0   -0.793  0.040  -0.867   -0.716      0.001      0.0    3926.0   
beta_p1  0.857  0.040   0.783    0.934      0.001      0.0    4879.0   
beta_p2  0.814  0.041   0.738    0.891      0.001      0.0    5634.0   
beta_rd  0.229  0.020   0.192    0.269      0.000      0.0    8749.0   
beta_ct  0.006  0.040  -0.071    0.081      0.001      0.0    6424.0   

         ess_tail  r_hat  
beta0      4934.0    1.0  
beta_p1    5291.0    1.0  
beta_p2    5268.0    1.0  
beta_rd    5258.0    1.0  
beta_ct    5289.0    1.0  


In [12]:
# 5. Bayesian model with VAGUE priors (for sensitivity analysis)

print("\n=== Fitting Bayesian model with vague priors ===")
with pm.Model() as model_vague:
    beta0 = pm.Normal("beta0", mu=0.0, sigma=10.0)
    beta_p1 = pm.Normal("beta_p1", mu=0.0, sigma=10.0)
    beta_p2 = pm.Normal("beta_p2", mu=0.0, sigma=10.0)
    beta_rd = pm.Normal("beta_rd", mu=0.0, sigma=10.0)
    beta_ct = pm.Normal("beta_ct", mu=0.0, sigma=10.0)
 
    logit_p = (beta0
               + beta_p1 * p1
               + beta_p2 * p2
               + beta_rd * rd
               + beta_ct * ct)
 
    y_obs = pm.Bernoulli("y_obs", logit_p=logit_p, observed=y)
 
    idata_vague = pm.sample(
        draws=2000,
        tune=1000,
        chains=4,
        random_seed=RANDOM_SEED,
        target_accept=0.9,
        return_inferencedata=True,
        progressbar=True,
    )
 
print(az.summary(idata_vague, var_names=["beta0", "beta_p1", "beta_p2",
                                         "beta_rd", "beta_ct"]))

Initializing NUTS using jitter+adapt_diag...



=== Fitting Bayesian model with vague priors ===


Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta0, beta_p1, beta_p2, beta_rd, beta_ct]


Output()

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 34 seconds.


          mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
beta0   -0.792  0.041  -0.874   -0.721      0.001      0.0    4399.0   
beta_p1  0.856  0.040   0.780    0.931      0.001      0.0    6025.0   
beta_p2  0.813  0.041   0.741    0.894      0.001      0.0    6106.0   
beta_rd  0.227  0.020   0.189    0.264      0.000      0.0    8788.0   
beta_ct  0.006  0.040  -0.070    0.083      0.001      0.0    6334.0   

         ess_tail  r_hat  
beta0      5344.0    1.0  
beta_p1    6192.0    1.0  
beta_p2    5462.0    1.0  
beta_rd    6056.0    1.0  
beta_ct    5529.0    1.0  


In [13]:
# 6. Frequentist comparison

print("\n=== Frequentist logistic regression ===")
freq_model = LogisticRegression(penalty=None, max_iter=1000)
freq_model.fit(X, y)
print(f"Intercept: {freq_model.intercept_[0]:.4f}")
print(f"Coefficients:")
for name, coef in zip(["pistol1", "pistol2", "rank_diff_std", "started_ct"],
                      freq_model.coef_[0]):
    print(f"  {name}: {coef:.4f}")
freq_acc = accuracy_score(y, freq_model.predict(X))


print(f"In-sample accuracy: {freq_acc:.3f}")


=== Frequentist logistic regression ===
Intercept: -0.7923
Coefficients:
  pistol1: 0.8560
  pistol2: 0.8136
  rank_diff_std: 0.2277
  started_ct: 0.0061
In-sample accuracy: 0.624


In [15]:
# 7. plots graphs and viz

print("\n=== Generating plots ===")
 
# Trace plots 
az.plot_trace(idata_inf, var_names=["beta0", "beta_p1", "beta_p2",
                                    "beta_rd", "beta_ct"])
plt.tight_layout()
plt.savefig("outputs/trace_informative.png", dpi=120, bbox_inches="tight")
plt.close()
 
# Posterior plots
az.plot_posterior(idata_inf, var_names=["beta_p1", "beta_p2",
                                        "beta_rd", "beta_ct"],
                  hdi_prob=0.95)
plt.tight_layout()
plt.savefig("outputs/posterior_informative.png", dpi=120, bbox_inches="tight")
plt.close()
 
# Side-by-side comparison of informative vs vague posteriors
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
params = ["beta_p1", "beta_p2", "beta_rd", "beta_ct"]
labels = ["Pistol 1 Win", "Pistol 2 Win", "Rank Diff (std)", "Started CT"]
for ax, param, label in zip(axes.flat, params, labels):
    inf_samples = idata_inf.posterior[param].values.flatten()
    vague_samples = idata_vague.posterior[param].values.flatten()
    ax.hist(inf_samples, bins=50, alpha=0.6, label="Informative", density=True)
    ax.hist(vague_samples, bins=50, alpha=0.6, label="Vague", density=True)
    ax.axvline(0, color="black", linestyle="--", alpha=0.5)
    ax.set_title(label)
    ax.legend()
plt.tight_layout()
plt.savefig("outputs/posterior_comparison.png", dpi=120, bbox_inches="tight")
plt.close()
 
# Posterior predictive check - predicted vs observed win rate by pistol outcome
ppc = idata_inf.posterior_predictive["y_obs"].values  # (chains, draws, N)
ppc_mean = ppc.mean(axis=(0, 1))
 
fig, ax = plt.subplots(figsize=(8, 5))
groups = [
    (0, 0, "Lost both"),
    (1, 0, "Won P1 only"),
    (0, 1, "Won P2 only"),
    (1, 1, "Won both"),
]
obs_rates, pred_rates, names = [], [], []
for a, b, name in groups:
    mask = (p1 == a) & (p2 == b)
    obs_rates.append(y[mask].mean())
    pred_rates.append(ppc_mean[mask].mean())
    names.append(name)
 
xpos = np.arange(len(names))
ax.bar(xpos - 0.2, obs_rates, width=0.4, label="Observed")
ax.bar(xpos + 0.2, pred_rates, width=0.4, label="Predicted (posterior mean)")
ax.set_xticks(xpos)
ax.set_xticklabels(names)
ax.set_ylabel("P(Team 1 wins map)")
ax.set_title("Posterior Predictive Check: Observed vs Predicted Win Rates")
ax.legend()
ax.axhline(0.5, color="black", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/ppc.png", dpi=120, bbox_inches="tight")
plt.close()


=== Generating plots ===


In [16]:
# 8. Odds ratios and interpretation

print("\n=== Odds Ratios (Informative Model) ===")
summary_inf = az.summary(idata_inf, var_names=["beta_p1", "beta_p2",
                                               "beta_rd", "beta_ct"],
                          hdi_prob=0.95)
for param in summary_inf.index:
    mean = summary_inf.loc[param, "mean"]
    lo = summary_inf.loc[param, "hdi_2.5%"]
    hi = summary_inf.loc[param, "hdi_97.5%"]
    print(f"  {param}: OR = {np.exp(mean):.2f} "
          f"(95% HDI: {np.exp(lo):.2f} - {np.exp(hi):.2f})")
 
# Save summaries
summary_inf.to_csv("outputs/summary_informative.csv")
az.summary(idata_vague, var_names=["beta0", "beta_p1", "beta_p2",
                                   "beta_rd", "beta_ct"],
           hdi_prob=0.95).to_csv("outputs/summary_vague.csv")
 
# Save trace data for reproducibility
idata_inf.to_netcdf("outputs/idata_informative.nc")
idata_vague.to_netcdf("outputs/idata_vague.nc")
 
print("\nDone. All outputs saved to ./outputs/")


=== Odds Ratios (Informative Model) ===
  beta_p1: OR = 2.36 (95% HDI: 2.18 - 2.55)
  beta_p2: OR = 2.26 (95% HDI: 2.09 - 2.45)
  beta_rd: OR = 1.26 (95% HDI: 1.21 - 1.31)
  beta_ct: OR = 1.01 (95% HDI: 0.93 - 1.09)

Done. All outputs saved to ./outputs/
